# Lab 4: Serving it on Cloud Run

[Open in Colab](https://colab.research.google.com/github/dominicdev/test-repo-11/blob/main/labs/lab4_deploy.ipynb)

## Lab 4: Serve it on Cloud Run (12 min)

The harness, state and coordination do not change when we leave the notebook. Only the entry point does:
a FastAPI service with `POST /runs` and `GET /runs/{id}`.

**No billing account?** Skip the deploy cells and set `SERVICE_URL` to the presenter's URL.

In [ ]:
# Setup: install the workshop package and load your Gemini API key.
import getpass, os, subprocess, sys

REPO = "git+https://github.com/dominicdev/test-repo-11.git@main"
try:
    import agent2027
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", REPO])
    import agent2027

if "GEMINI_API_KEY" not in os.environ:
    try:  # Colab: Secrets panel (key icon) -> add GEMINI_API_KEY
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key (from AI Studio): ")

from agent2027 import config
print("model:", config.MODEL, "| search mode:", config.SEARCH_MODE)

### 1. Deploy (needs a project with billing)
This uses instance-based billing (`--no-cpu-throttling`) and `--min-instances 1`: the run keeps working
in background threads after the POST has returned, so CPU must not be throttled.

In [ ]:
# Clone the repo so we have deploy/cloudrun/. Set DEPLOY = True to run it.
DEPLOY = False
PROJECT = "your-gcp-project"   # <- change me

if DEPLOY:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("devfest-agents-2027"):
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/dominicdev/test-repo-11.git"])
    env = {**os.environ, "PROJECT": PROJECT}
    r = subprocess.run(["bash", "deploy/cloudrun/deploy.sh"], cwd="devfest-agents-2027",
                       env=env, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    SERVICE_URL = r.stdout.strip().splitlines()[-1]

In [ ]:
SERVICE_URL = globals().get("SERVICE_URL") or "https://PRESENTER-URL.run.app"  # <- or the presenter's URL
import httpx
print(httpx.get(f"{SERVICE_URL}/healthz", timeout=30).json())

### 2. Start a run and poll it

In [ ]:
import time
goal = "Compare three popular vector databases for a small startup."
run_id = httpx.post(f"{SERVICE_URL}/runs", json={"goal": goal, "n_workers": 3}, timeout=30).json()["run_id"]
print("run_id:", run_id)

while True:
    s = httpx.get(f"{SERVICE_URL}/runs/{run_id}", timeout=30).json()
    print(s["status"], [(t["role"], t["status"]) for t in s["tasks"]])
    if s["status"] in ("done", "failed"):
        break
    time.sleep(10)
print("\n", s["result"])

### 3. Same state, same task board
The service writes to Firestore, so the live view shows it exactly as it showed the notebook run.

### Cloud Run vs Agent Runtime
- **Cloud Run:** you own the container; works with any harness. Watch CPU allocation and request timeouts for background work.
- **Agent Runtime:** managed runtime for agents with Sessions, Memory Bank, long-running operations and identity built in.
- Rule of thumb: Cloud Run for control and portability; Agent Runtime when you want the agent services managed for you.

In production, replace the in-process threads with Cloud Run jobs or Cloud Tasks.